# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_01 — Discrete Martingale Pricing in a Binomial Model

### Research question

How can derivative prices be obtained in a discrete-time binomial market using no-arbitrage, replication, and risk-neutral martingale measures?

This notebook introduces the core Shreve-style logic behind discrete-time asset pricing:

1. define a one-period binomial market;
2. derive the no-arbitrage condition;
3. construct the risk-neutral probability;
4. show that the discounted stock price is a martingale under $\mathbb Q$;
5. price a derivative by risk-neutral expectation;
6. recover the same price by replication;
7. extend the method to a multi-period tree;
8. compute the replicating delta strategy;
9. validate put-call parity and terminal replication;
10. connect the finite-state theorem to modern research directions.

The goal is to make the fundamental idea computationally explicit:

> In an arbitrage-free complete binomial model, derivative prices are not determined by the real-world probability of an up move. They are determined by the unique risk-neutral measure under which discounted tradable prices are martingales.

## 1. Motivation

The previous Phase 0 notebooks introduced martingales, Brownian motion, linear algebra, optimisation, and Python/C++ implementation.

This notebook now uses those foundations for actual derivative pricing.

The key objects are:

- a risky stock $S$;
- a risk-free money market account $B$;
- a derivative payoff $V_T = g(S_T)$;
- a trading strategy that replicates the derivative payoff;
- a probability measure $\mathbb Q$ under which discounted asset prices are martingales.

The discrete binomial model is important because it is small enough to solve exactly but rich enough to contain the central logic of mathematical finance:

$$
\text{No arbitrage}
\quad \Longleftrightarrow \quad
\text{existence of a risk-neutral martingale measure}
$$

In a complete one-stock binomial model, the martingale measure is unique, so derivative prices are unique.

## 2. One-period binomial market

At time $0$, the stock price is:

$$
S_0
$$

At time $1$, the stock can move up or down:

$$
S_1 =
\begin{cases}
uS_0, & \text{up state} \\
dS_0, & \text{down state}
\end{cases}
$$

where:

$$
u > d > 0
$$

The risk-free account grows from:

$$
B_0 = 1
$$

to:

$$
B_1 = 1+r
$$

where $r$ is the one-period risk-free rate.

The no-arbitrage condition is:

$$
d < 1+r < u
$$

If $1+r \geq u$, the stock is dominated by the bond.  
If $1+r \leq d$, the stock dominates the bond.  
Only when the bond return lies between the down and up stock returns can the model be arbitrage-free.

## 3. Risk-neutral probability

We want a probability $q$ such that the discounted stock price is a martingale.

The martingale condition is:

$$
\begin{aligned}
S_0 &= \frac{1}{1+r} \mathbb E^{\mathbb Q}[S_1]
\end{aligned}
$$

Substitute the binomial states:

$$
\begin{aligned}
S_0 &= \frac{1}{1+r} \left( q u S_0 + (1-q)dS_0 \right)
\end{aligned}
$$

Cancel $S_0$:

$$
1+r = q u + (1-q)d
$$

Solving for $q$:

$$
q = \frac{1+r-d}{u-d}
$$

The no-arbitrage condition:

$$
d < 1+r < u
$$

is exactly the condition that:

$$
0 < q < 1
$$

So the risk-neutral probability is a genuine probability if and only if the one-period binomial model has no arbitrage.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

In [ ]:
@dataclass(frozen=True)
class OnePeriodBinomialModel:
    """
    One-period binomial model.

    Parameters
    ----------
    s0:
        Initial stock price.

    u:
        Up multiplier.

    d:
        Down multiplier.

    r:
        One-period risk-free rate.

    p:
        Physical probability of an up move under P.
        This is not used for arbitrage-free pricing.
    """
    s0: float
    u: float
    d: float
    r: float
    p: float

    @property
    def gross_risk_free_return(self) -> float:
        return 1.0 + self.r

    @property
    def q(self) -> float:
        """
        Risk-neutral probability.
        """
        return (self.gross_risk_free_return - self.d) / (self.u - self.d)

    def validate_no_arbitrage(self) -> None:
        """
        Raise an error if the binomial model violates no-arbitrage.
        """
        if not (self.d < self.gross_risk_free_return < self.u):
            raise ValueError("No-arbitrage condition violated. Need d < 1+r < u.")

        if not (0.0 < self.q < 1.0):
            raise ValueError("Risk-neutral probability is not in (0, 1).")

    def stock_states(self) -> dict:
        """
        Return up/down stock prices.
        """
        return {
            "S0": self.s0,
            "S_up": self.u * self.s0,
            "S_down": self.d * self.s0
        }


model = OnePeriodBinomialModel(
    s0=100.0,
    u=1.12,
    d=0.92,
    r=0.03,
    p=0.60
)

model.validate_no_arbitrage()

model.q, model.stock_states()

## 4. Physical probability versus risk-neutral probability

The physical probability $p$ describes the real-world probability of an up move under $\mathbb P$.

The risk-neutral probability $q$ is chosen so that the discounted stock is a martingale under $\mathbb Q$.

These are usually not equal.

Under $\mathbb P$, the expected gross stock return is:

$$
\begin{aligned}
\mathbb E^{\mathbb P}\left[\frac{S_1}{S_0}\right] &= pu + (1-p)d
\end{aligned}
$$

Under $\mathbb Q$, the expected gross stock return is:

$$
\begin{aligned}
\mathbb E^{\mathbb Q}\left[\frac{S_1}{S_0}\right] &= qu + (1-q)d \\
&= 1+r
\end{aligned}
$$

The risk-neutral measure is not claiming investors are risk-neutral in the real world. It is a pricing measure created by no-arbitrage.

In [ ]:
def one_period_measure_comparison(model: OnePeriodBinomialModel) -> pd.Series:
    """
    Compare expected gross returns under physical and risk-neutral measures.
    """
    expected_gross_return_p = model.p * model.u + (1 - model.p) * model.d
    expected_gross_return_q = model.q * model.u + (1 - model.q) * model.d

    return pd.Series({
        "physical_probability_p": model.p,
        "risk_neutral_probability_q": model.q,
        "expected_gross_stock_return_under_P": expected_gross_return_p,
        "expected_gross_stock_return_under_Q": expected_gross_return_q,
        "gross_risk_free_return": model.gross_risk_free_return,
        "discounted_stock_expectation_under_Q": (
            model.s0 * expected_gross_return_q / model.gross_risk_free_return
        )
    })


one_period_measure_comparison(model)

## 5. Pricing by risk-neutral expectation

Let a derivative payoff at time $1$ be:

$$
V_1 =
\begin{cases}
V_u, & \text{up state} \\
V_d, & \text{down state}
\end{cases}
$$

The no-arbitrage price is:

$$
\begin{aligned}
V_0 &= \frac{1}{1+r} \mathbb E^{\mathbb Q}[V_1]
\end{aligned}
$$

So:

$$
\begin{aligned}
V_0 &= \frac{1}{1+r} \left( qV_u + (1-q)V_d \right)
\end{aligned}
$$

For a European call with strike $K$:

$$
V_u = \max(uS_0-K,0)
$$

$$
V_d = \max(dS_0-K,0)
$$

In [ ]:
def call_payoff(stock_price: float | np.ndarray, strike: float) -> float | np.ndarray:
    """
    European call payoff.
    """
    return np.maximum(np.asarray(stock_price) - strike, 0.0)


def put_payoff(stock_price: float | np.ndarray, strike: float) -> float | np.ndarray:
    """
    European put payoff.
    """
    return np.maximum(strike - np.asarray(stock_price), 0.0)


def one_period_risk_neutral_price(
    model: OnePeriodBinomialModel,
    payoff_func,
    **payoff_kwargs
) -> dict:
    """
    Price a one-period derivative by risk-neutral expectation.
    """
    model.validate_no_arbitrage()

    s_up = model.s0 * model.u
    s_down = model.s0 * model.d

    v_up = float(payoff_func(s_up, **payoff_kwargs))
    v_down = float(payoff_func(s_down, **payoff_kwargs))

    price = (model.q * v_up + (1 - model.q) * v_down) / model.gross_risk_free_return

    return {
        "S_up": s_up,
        "S_down": s_down,
        "V_up": v_up,
        "V_down": v_down,
        "risk_neutral_probability_q": model.q,
        "price": price
    }


strike = 100.0

call_price_rn = one_period_risk_neutral_price(
    model,
    call_payoff,
    strike=strike
)

call_price_rn

## 6. Pricing by replication

The derivative can also be priced by constructing a self-financing portfolio of:

- $\Delta$ shares of stock;
- $\beta$ units of the money market account.

At time $1$, the replicating portfolio must satisfy:

$$
\Delta uS_0 + \beta(1+r) = V_u
$$

$$
\Delta dS_0 + \beta(1+r) = V_d
$$

Subtract the equations:

$$
\Delta (u-d)S_0 = V_u - V_d
$$

So:

$$
\Delta = \frac{V_u - V_d}{(u-d)S_0}
$$

Then:

$$
\begin{aligned}
\beta &= \frac{V_u - \Delta uS_0}{1+r}
\end{aligned}
$$

The time-zero cost of the replicating portfolio is:

$$
V_0 = \Delta S_0 + \beta
$$

In a no-arbitrage complete one-period binomial model, this must equal the risk-neutral expectation price.

In [ ]:
def one_period_replicating_portfolio(
    model: OnePeriodBinomialModel,
    payoff_func,
    **payoff_kwargs
) -> dict:
    """
    Compute one-period replicating portfolio for a derivative.
    """
    model.validate_no_arbitrage()

    s_up = model.s0 * model.u
    s_down = model.s0 * model.d

    v_up = float(payoff_func(s_up, **payoff_kwargs))
    v_down = float(payoff_func(s_down, **payoff_kwargs))

    delta = (v_up - v_down) / (s_up - s_down)
    beta = (v_up - delta * s_up) / model.gross_risk_free_return

    price = delta * model.s0 + beta

    up_replication = delta * s_up + beta * model.gross_risk_free_return
    down_replication = delta * s_down + beta * model.gross_risk_free_return

    return {
        "delta": delta,
        "beta_cash_account_units": beta,
        "replication_price": price,
        "up_replication_value": up_replication,
        "down_replication_value": down_replication,
        "V_up": v_up,
        "V_down": v_down,
        "up_replication_error": up_replication - v_up,
        "down_replication_error": down_replication - v_down
    }


call_replication = one_period_replicating_portfolio(
    model,
    call_payoff,
    strike=strike
)

call_replication

In [ ]:
comparison_one_period = pd.Series({
    "risk_neutral_price": call_price_rn["price"],
    "replication_price": call_replication["replication_price"],
    "absolute_difference": abs(call_price_rn["price"] - call_replication["replication_price"]),
    "delta": call_replication["delta"],
    "beta": call_replication["beta_cash_account_units"]
})

comparison_one_period

## 7. Put-call parity in one period

For European call and put options with the same strike $K$ and maturity, put-call parity is:

$$
C_0 - P_0 = S_0 - \frac{K}{1+r}
$$

This follows from comparing two portfolios:

1. call plus discounted strike cash;
2. put plus stock.

Both have the same terminal payoff:

$$
\begin{aligned}
\max(S_1-K,0)+K &= \max(K-S_1,0)+S_1
\end{aligned}
$$

So their time-zero prices must match under no arbitrage.

In [ ]:
call_price = one_period_risk_neutral_price(
    model,
    call_payoff,
    strike=strike
)["price"]

put_price = one_period_risk_neutral_price(
    model,
    put_payoff,
    strike=strike
)["price"]

put_call_parity_check = pd.Series({
    "call_price": call_price,
    "put_price": put_price,
    "C_minus_P": call_price - put_price,
    "S0_minus_discounted_K": model.s0 - strike / model.gross_risk_free_return,
    "parity_error": (call_price - put_price) - (model.s0 - strike / model.gross_risk_free_return)
})

put_call_parity_check

## 8. Multi-period binomial model

Now extend to $N$ periods.

At each step:

$$
S_{n+1} =
\begin{cases}
uS_n, & \text{up} \\
dS_n, & \text{down}
\end{cases}
$$

The money market account grows as:

$$
B_n = (1+r)^n
$$

The discounted stock price is:

$$
\tilde S_n = \frac{S_n}{B_n}
$$

Under the risk-neutral measure $\mathbb Q$, the discounted stock price should satisfy:

$$
\begin{aligned}
\mathbb E^{\mathbb Q}[\tilde S_{n+1} \mid \mathcal F_n] &= \tilde S_n
\end{aligned}
$$

The same risk-neutral probability is used at every node:

$$
q = \frac{1+r-d}{u-d}
$$

if the model parameters are constant.

In [ ]:
@dataclass(frozen=True)
class MultiPeriodBinomialModel:
    """
    Multi-period binomial model with constant u, d, r.
    """
    s0: float
    u: float
    d: float
    r: float
    n_steps: int
    p: float = 0.5

    @property
    def gross_risk_free_return(self) -> float:
        return 1.0 + self.r

    @property
    def q(self) -> float:
        return (self.gross_risk_free_return - self.d) / (self.u - self.d)

    def validate_no_arbitrage(self) -> None:
        if not (self.d < self.gross_risk_free_return < self.u):
            raise ValueError("No-arbitrage condition violated. Need d < 1+r < u.")

        if not (0.0 < self.q < 1.0):
            raise ValueError("Risk-neutral probability is not in (0,1).")

In [ ]:
def build_stock_tree(model: MultiPeriodBinomialModel) -> list[np.ndarray]:
    """
    Build recombining binomial stock price tree.

    tree[n][j] is the stock price at time n after j up moves.
    """
    model.validate_no_arbitrage()

    tree = []

    for n in range(model.n_steps + 1):
        prices = np.array([
            model.s0 * (model.u ** j) * (model.d ** (n - j))
            for j in range(n + 1)
        ], dtype=float)

        tree.append(prices)

    return tree


multi_model = MultiPeriodBinomialModel(
    s0=100.0,
    u=1.08,
    d=0.94,
    r=0.02,
    n_steps=5,
    p=0.58
)

stock_tree = build_stock_tree(multi_model)

stock_tree

## 9. Backward induction for European options

At maturity $N$, define the payoff:

$$
V_N = g(S_N)
$$

Then work backward:

$$
\begin{aligned}
V_n &= \frac{1}{1+r} \left[ q V_{n+1}^{u} + (1-q)V_{n+1}^{d} \right]
\end{aligned}
$$

This is dynamic programming on the binomial tree.

At each node, the value is the discounted risk-neutral conditional expectation of next-period values.

In [ ]:
def price_european_option_tree(
    model: MultiPeriodBinomialModel,
    payoff_func,
    **payoff_kwargs
) -> dict:
    """
    Price a European derivative by backward induction.
    """
    model.validate_no_arbitrage()

    stock_tree = build_stock_tree(model)

    value_tree = [None] * (model.n_steps + 1)
    value_tree[-1] = payoff_func(stock_tree[-1], **payoff_kwargs).astype(float)

    for n in range(model.n_steps - 1, -1, -1):
        next_values = value_tree[n + 1]
        current_values = (
            model.q * next_values[1:]
            + (1 - model.q) * next_values[:-1]
        ) / model.gross_risk_free_return

        value_tree[n] = current_values

    return {
        "price": float(value_tree[0][0]),
        "stock_tree": stock_tree,
        "value_tree": value_tree,
        "risk_neutral_probability_q": model.q
    }


multi_call = price_european_option_tree(
    multi_model,
    call_payoff,
    strike=100.0
)

multi_call["price"]

In [ ]:
def tree_to_dataframe(tree: list[np.ndarray], value_name: str) -> pd.DataFrame:
    """
    Convert a recombining tree into a long DataFrame.
    """
    rows = []

    for n, values in enumerate(tree):
        for j, value in enumerate(values):
            rows.append({
                "time_step": n,
                "up_moves": j,
                value_name: float(value)
            })

    return pd.DataFrame(rows)


stock_tree_df = tree_to_dataframe(multi_call["stock_tree"], "stock_price")
value_tree_df = tree_to_dataframe(multi_call["value_tree"], "option_value")

tree_df = stock_tree_df.merge(
    value_tree_df,
    on=["time_step", "up_moves"],
    how="left"
)

tree_df.head(10)

## 10. Delta tree and replication

At each non-terminal node, the local hedge ratio is:

$$
\begin{aligned}
\Delta_n &= \frac{V_{n+1}^{u} - V_{n+1}^{d}} {S_{n+1}^{u} - S_{n+1}^{d}}
\end{aligned}
$$

The cash account position is:

$$
\begin{aligned}
\beta_n &= \frac{V_{n+1}^{u} - \Delta_n S_{n+1}^{u}} {1+r}
\end{aligned}
$$

This creates a one-step replicating portfolio from each node.

The delta changes through time and across states.

In [ ]:
def compute_delta_beta_trees(
    model: MultiPeriodBinomialModel,
    stock_tree: list[np.ndarray],
    value_tree: list[np.ndarray]
) -> dict:
    """
    Compute node-by-node delta and cash-account positions.
    """
    delta_tree = []
    beta_tree = []

    for n in range(model.n_steps):
        deltas = []
        betas = []

        for j in range(n + 1):
            s_down = stock_tree[n + 1][j]
            s_up = stock_tree[n + 1][j + 1]

            v_down = value_tree[n + 1][j]
            v_up = value_tree[n + 1][j + 1]

            delta = (v_up - v_down) / (s_up - s_down)
            beta = (v_up - delta * s_up) / model.gross_risk_free_return

            deltas.append(delta)
            betas.append(beta)

        delta_tree.append(np.array(deltas))
        beta_tree.append(np.array(betas))

    return {
        "delta_tree": delta_tree,
        "beta_tree": beta_tree
    }


hedge_trees = compute_delta_beta_trees(
    multi_model,
    multi_call["stock_tree"],
    multi_call["value_tree"]
)

delta_df = tree_to_dataframe(hedge_trees["delta_tree"], "delta")
beta_df = tree_to_dataframe(hedge_trees["beta_tree"], "beta")

hedge_df = delta_df.merge(beta_df, on=["time_step", "up_moves"])

hedge_df.head(10)

## 11. Terminal replication test

A correct delta/cash strategy should replicate the option payoff at maturity along every path.

We simulate all possible up/down paths through the tree and apply the dynamic hedge.

At each node:

$$
V_n = \Delta_n S_n + \beta_n
$$

Then the portfolio evolves self-financing to the next node.

At maturity, the final value should equal:

$$
g(S_N)
$$

for every possible path.

In [ ]:
def enumerate_binomial_paths(n_steps: int) -> list[list[int]]:
    """
    Enumerate all up/down paths.

    Down = 0, Up = 1.
    """
    if n_steps == 0:
        return [[]]

    previous = enumerate_binomial_paths(n_steps - 1)

    return [path + [0] for path in previous] + [path + [1] for path in previous]


def test_dynamic_replication(
    model: MultiPeriodBinomialModel,
    payoff_func,
    stock_tree: list[np.ndarray],
    value_tree: list[np.ndarray],
    delta_tree: list[np.ndarray],
    beta_tree: list[np.ndarray],
    **payoff_kwargs
) -> pd.DataFrame:
    """
    Test dynamic replication along all binomial paths.
    """
    paths = enumerate_binomial_paths(model.n_steps)
    rows = []

    for path_id, path in enumerate(paths):
        up_moves = 0
        portfolio_value = float(value_tree[0][0])

        for n, move in enumerate(path):
            delta = delta_tree[n][up_moves]
            beta = beta_tree[n][up_moves]

            if move == 1:
                next_up_moves = up_moves + 1
            else:
                next_up_moves = up_moves

            next_stock = stock_tree[n + 1][next_up_moves]
            portfolio_value = delta * next_stock + beta * model.gross_risk_free_return

            up_moves = next_up_moves

        terminal_stock = stock_tree[model.n_steps][up_moves]
        payoff = float(payoff_func(terminal_stock, **payoff_kwargs))

        rows.append({
            "path_id": path_id,
            "path": "".join("U" if move == 1 else "D" for move in path),
            "terminal_stock": terminal_stock,
            "replicating_portfolio_terminal_value": portfolio_value,
            "payoff": payoff,
            "replication_error": portfolio_value - payoff
        })

    return pd.DataFrame(rows)


replication_test_df = test_dynamic_replication(
    model=multi_model,
    payoff_func=call_payoff,
    stock_tree=multi_call["stock_tree"],
    value_tree=multi_call["value_tree"],
    delta_tree=hedge_trees["delta_tree"],
    beta_tree=hedge_trees["beta_tree"],
    strike=100.0
)

replication_test_df.head()

In [ ]:
replication_error_summary = pd.Series({
    "max_abs_replication_error": replication_test_df["replication_error"].abs().max(),
    "mean_abs_replication_error": replication_test_df["replication_error"].abs().mean(),
    "num_paths": len(replication_test_df)
})

replication_error_summary

## 12. Martingale check under $\mathbb Q$

The discounted stock price should be a martingale under $\mathbb Q$:

$$
\begin{aligned}
\mathbb E^{\mathbb Q} \left[ \frac{S_{n+1}}{(1+r)^{n+1}} \mid \mathcal F_n \right] &= \frac{S_n}{(1+r)^n}
\end{aligned}
$$

We verify this numerically at every node.

In [ ]:
def martingale_check_stock_tree(model: MultiPeriodBinomialModel) -> pd.DataFrame:
    """
    Check discounted stock martingale condition at every non-terminal node.
    """
    stock_tree = build_stock_tree(model)
    rows = []

    for n in range(model.n_steps):
        for j in range(n + 1):
            s_now = stock_tree[n][j]
            s_down = stock_tree[n + 1][j]
            s_up = stock_tree[n + 1][j + 1]

            discounted_now = s_now / (model.gross_risk_free_return ** n)

            expected_discounted_next = (
                model.q * s_up
                + (1 - model.q) * s_down
            ) / (model.gross_risk_free_return ** (n + 1))

            rows.append({
                "time_step": n,
                "up_moves": j,
                "discounted_now": discounted_now,
                "expected_discounted_next_under_Q": expected_discounted_next,
                "martingale_error": expected_discounted_next - discounted_now
            })

    return pd.DataFrame(rows)


martingale_check_df = martingale_check_stock_tree(multi_model)

martingale_check_df.head()

In [ ]:
martingale_check_summary = pd.Series({
    "max_abs_martingale_error": martingale_check_df["martingale_error"].abs().max(),
    "mean_abs_martingale_error": martingale_check_df["martingale_error"].abs().mean()
})

martingale_check_summary

## 13. Physical measure does not price the option

Now compare two expected discounted payoff calculations:

$$
\frac{1}{(1+r)^N}\mathbb E^{\mathbb Q}[g(S_N)]
$$

and:

$$
\frac{1}{(1+r)^N}\mathbb E^{\mathbb P}[g(S_N)]
$$

Only the $\mathbb Q$-expectation gives the no-arbitrage price.

The physical probability $p$ may be useful for forecasting or risk management, but not for arbitrage-free derivative pricing in this complete model.

In [ ]:
from math import comb


def terminal_state_probabilities(n_steps: int, prob_up: float) -> pd.DataFrame:
    """
    Binomial probabilities by number of up moves.
    """
    rows = []

    for j in range(n_steps + 1):
        probability = comb(n_steps, j) * (prob_up ** j) * ((1 - prob_up) ** (n_steps - j))
        rows.append({
            "up_moves": j,
            "probability": probability
        })

    return pd.DataFrame(rows)


def discounted_expected_payoff(
    model: MultiPeriodBinomialModel,
    payoff_func,
    prob_up: float,
    **payoff_kwargs
) -> float:
    """
    Discounted expected payoff under a chosen up probability.
    """
    stock_tree = build_stock_tree(model)
    terminal_stock = stock_tree[-1]
    payoff = payoff_func(terminal_stock, **payoff_kwargs)

    probs = terminal_state_probabilities(model.n_steps, prob_up)["probability"].to_numpy()

    return float(np.sum(probs * payoff) / (model.gross_risk_free_return ** model.n_steps))


price_under_q = discounted_expected_payoff(
    multi_model,
    call_payoff,
    prob_up=multi_model.q,
    strike=100.0
)

discounted_expectation_under_p = discounted_expected_payoff(
    multi_model,
    call_payoff,
    prob_up=multi_model.p,
    strike=100.0
)

physical_vs_rn = pd.Series({
    "price_by_backward_induction": multi_call["price"],
    "discounted_expected_payoff_under_Q": price_under_q,
    "discounted_expected_payoff_under_P": discounted_expectation_under_p,
    "physical_probability_p": multi_model.p,
    "risk_neutral_probability_q": multi_model.q
})

physical_vs_rn

## 14. Visualising price, option value, and delta trees

Tree visualisation helps make dynamic replication more concrete.

Each node has:

- stock price;
- option value;
- hedge delta, if non-terminal.

In [ ]:
def plot_tree_nodes(tree: list[np.ndarray], title: str, value_label: str) -> None:
    """
    Plot a recombining tree as a scatter plot.
    """
    xs = []
    ys = []
    labels = []

    for n, values in enumerate(tree):
        for j, value in enumerate(values):
            xs.append(n)
            ys.append(j - n / 2)
            labels.append(value)

    plt.figure(figsize=(10, 6))
    plt.scatter(xs, ys)

    for x, y, label in zip(xs, ys, labels):
        plt.text(x, y, f"{label:.2f}", ha="center", va="bottom", fontsize=8)

    plt.title(title)
    plt.xlabel("Time step")
    plt.ylabel("State position")
    plt.grid(True, alpha=0.3)
    plt.show()


plot_tree_nodes(multi_call["stock_tree"], "Stock Price Tree", "S")
plot_tree_nodes(multi_call["value_tree"], "European Call Value Tree", "V")

In [ ]:
plot_tree_nodes(hedge_trees["delta_tree"], "Delta Hedge Tree", "Delta")

## 15. Arbitrage examples when no-arbitrage fails

The condition:

$$
d < 1+r < u
$$

is not decorative.

If it fails, there is an arbitrage.

### Case 1: $1+r \geq u$

The bond return is at least as good as the best stock return.  
Short the stock and buy the bond.

### Case 2: $1+r \leq d$

The stock return is at least as good as the bond return in every state.  
Borrow at the risk-free rate and buy the stock.

The risk-neutral probability formula reveals the same issue because $q$ falls outside $[0,1]$.

In [ ]:
def diagnose_one_period_arbitrage(model: OnePeriodBinomialModel) -> pd.Series:
    """
    Diagnose no-arbitrage condition and risk-neutral probability.
    """
    q = model.q

    if model.gross_risk_free_return >= model.u:
        arbitrage_description = "Bond weakly dominates stock. Short stock and buy bond."
        no_arbitrage = False
    elif model.gross_risk_free_return <= model.d:
        arbitrage_description = "Stock weakly dominates bond. Borrow and buy stock."
        no_arbitrage = False
    else:
        arbitrage_description = "No one-period arbitrage detected."
        no_arbitrage = True

    return pd.Series({
        "d": model.d,
        "1_plus_r": model.gross_risk_free_return,
        "u": model.u,
        "risk_neutral_probability_q": q,
        "q_in_unit_interval": 0 <= q <= 1,
        "no_arbitrage_condition_holds": no_arbitrage,
        "diagnosis": arbitrage_description
    })


bad_model_1 = OnePeriodBinomialModel(
    s0=100,
    u=1.04,
    d=0.95,
    r=0.06,
    p=0.5
)

bad_model_2 = OnePeriodBinomialModel(
    s0=100,
    u=1.15,
    d=1.04,
    r=0.02,
    p=0.5
)

pd.DataFrame({
    "valid_model": diagnose_one_period_arbitrage(model),
    "bad_model_bond_dominates": diagnose_one_period_arbitrage(bad_model_1),
    "bad_model_stock_dominates": diagnose_one_period_arbitrage(bad_model_2),
})

## 16. Finite-market theorem intuition

In finite discrete-time markets, the Fundamental Theorems of Asset Pricing can be stated informally as:

### First theorem

A market has no arbitrage if and only if there exists an equivalent martingale measure.

### Second theorem

If the market is complete, then the equivalent martingale measure is unique.

In the one-stock binomial model:

- there are two states;
- stock and bond provide two tradable instruments;
- every one-period payoff can be replicated;
- the market is complete;
- the risk-neutral probability $q$ is unique.

This is why the derivative price is unique.

## 17. Limitations

This notebook deliberately uses a clean binomial model.

### 17.1 Only two states per step

Real markets do not move only up or down by fixed multipliers.

The binomial model is a discretisation, not a literal description of price dynamics.

### 17.2 Constant parameters

The notebook uses constant $u$, $d$, and $r$.

Real markets have stochastic volatility, stochastic rates, jumps, liquidity constraints, and regime changes.

### 17.3 Frictionless trading

The replication argument assumes:

- no transaction costs;
- no bid-ask spread;
- no market impact;
- continuous ability to trade at node prices;
- unlimited borrowing/lending at the risk-free rate;
- no margin constraints.

These assumptions break down in execution-aware trading systems.

### 17.4 Complete market

The binomial model is complete because every payoff can be replicated using stock and bond.

Real markets are often incomplete, especially with jumps, stochastic volatility, transaction costs, and liquidity constraints.

### 17.5 Risk-neutral pricing is not forecasting

The risk-neutral probability $q$ is not a forecast probability.

It is a pricing device implied by no-arbitrage and replication.

## 18. Research frontier and current directions

Discrete-time martingale pricing remains relevant because real hedging is discrete, costly, and constrained.

### 18.1 Martingale optimal transport

Classical binomial pricing assumes a specific stochastic model.

Martingale Optimal Transport asks a more robust question:

> What are the no-arbitrage price bounds using only partial information, such as observed option prices or marginal distributions?

This connects the martingale constraint from this notebook to model-independent pricing and robust hedging.

A future notebook could solve a small two-period martingale transport problem and compare robust price bounds with binomial prices.

### 18.2 Pricing under transaction costs

The binomial model assumes frictionless self-financing replication.

With transaction costs, exact replication may become impossible or too expensive.

Modern research studies super-replication, no-transaction bands, utility indifference pricing, and frictional martingale transport.

A future notebook could add proportional transaction costs to the binomial hedge and show how perfect replication breaks.

### 18.3 Deep hedging

Deep hedging uses neural networks or reinforcement learning to learn hedging strategies under transaction costs, liquidity constraints, market impact, and risk objectives.

This is a modern extension of the discrete hedging idea: instead of deriving a closed-form delta, the hedge is learned from simulated or historical scenarios.

A future notebook could compare binomial delta hedging with a simple learned hedge under transaction costs.

### 18.4 Robust hedging and model risk

If the model parameters $u$, $d$, $r$, or volatility are uncertain, a single price may be misleading.

Robust pricing asks for price intervals or hedges that work across a set of plausible models.

A future notebook could stress the binomial price across many $u,d$ values and visualise model-risk sensitivity.

### 18.5 Discrete-time calibration

In practice, binomial and trinomial trees are calibrated to observed volatility surfaces or local volatility.

A future notebook could calibrate tree parameters to match an implied volatility smile.

### 18.6 Connection to stochastic control

Dynamic hedging is a control problem.

At each time step, the trader chooses a portfolio based on current information.

With frictions, constraints, and risk preferences, the problem becomes a stochastic control or dynamic programming problem.

A future notebook could formulate hedging as a finite-horizon dynamic program.

## 19. Suggested follow-up notebooks

This notebook naturally leads to:

1. **02_02_crr_binomial_lattice**  
   Implementing Cox-Ross-Rubinstein trees and convergence behaviour.

2. **02_03_black_scholes_merton_pde**  
   Moving from discrete binomial pricing to continuous-time PDE pricing.

3. **02_04_implied_volatility_root_finding**  
   Inverting option prices to recover implied volatility.

4. **02_06_monte_carlo_gbm_and_fat_tails**  
   Pricing by simulation rather than tree recursion.

5. **02_08_dynamic_delta_hedging_simulation**  
   Testing replication when hedging is discrete and markets move continuously.

6. **05_03_transaction_costs_slippage_latency**  
   Adding realistic execution costs to theoretical hedges.

7. **06_deep_hedging_with_transaction_costs**  
   Learning hedging strategies under frictions.

8. **07_02_volatility_surface_pricing_and_alpha**  
   Connecting pricing models to observed option surfaces.

## 20. Saving pricing outputs

The notebook saves:

1. one-period pricing comparison;
2. multi-period stock/value/hedge trees;
3. martingale check table;
4. terminal replication test;
5. pricing manifest.

These files make the notebook output auditable and reusable.

In [ ]:
output_dir = Path("data/processed/discrete_martingale_pricing_v1")
output_dir.mkdir(parents=True, exist_ok=True)

one_period_path = output_dir / "one_period_pricing_comparison.csv"
tree_path = output_dir / "multi_period_stock_value_tree.csv"
hedge_path = output_dir / "multi_period_delta_beta_tree.csv"
martingale_path = output_dir / "discounted_stock_martingale_check.csv"
replication_path = output_dir / "terminal_replication_test.csv"
manifest_path = output_dir / "manifest.json"

one_period_output = pd.DataFrame([
    {
        **{f"model_{key}": value for key, value in asdict(model).items()},
        **call_price_rn,
        **{f"replication_{key}": value for key, value in call_replication.items()}
    }
])

one_period_output.to_csv(one_period_path, index=False)
tree_df.to_csv(tree_path, index=False)
hedge_df.to_csv(hedge_path, index=False)
martingale_check_df.to_csv(martingale_path, index=False)
replication_test_df.to_csv(replication_path, index=False)

manifest = {
    "dataset_name": "discrete_martingale_pricing_outputs",
    "schema_version": "discrete_martingale_pricing_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "one_period_model": asdict(model),
    "multi_period_model": asdict(multi_model),
    "payoff": "European call",
    "strike": 100.0,
    "one_period_call_price": float(call_price_rn["price"]),
    "multi_period_call_price": float(multi_call["price"]),
    "max_abs_martingale_error": float(martingale_check_summary["max_abs_martingale_error"]),
    "max_abs_replication_error": float(replication_error_summary["max_abs_replication_error"]),
    "notes": [
        "Prices are computed under the risk-neutral measure Q.",
        "Physical probability p is shown for comparison but is not used for no-arbitrage pricing.",
        "The model is frictionless, complete, and recombining."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

one_period_path, tree_path, hedge_path, martingale_path, replication_path, manifest_path

## 21. Summary

This notebook implemented discrete martingale pricing in a binomial model.

It showed that:

1. no-arbitrage requires:

$$
d < 1+r < u
$$

2. the risk-neutral probability is:

$$
q = \frac{1+r-d}{u-d}
$$

3. discounted stock prices are martingales under $\mathbb Q$;
4. derivative prices are discounted $\mathbb Q$-expectations;
5. the same price is obtained by replication;
6. multi-period prices are computed by backward induction;
7. the delta hedge replicates the terminal payoff along every binomial path;
8. physical probabilities are not used for arbitrage-free pricing.

The key mathematical takeaway is:

> No-arbitrage changes pricing from a forecasting problem into a martingale problem under an equivalent pricing measure.

The key financial takeaway is:

> A derivative price is the cost of replication in a complete market, not the real-world expected payoff discounted using the physical probability.

## 22. Further reading

### Core textbook foundations

- Shreve, S. E. *Stochastic Calculus for Finance I: The Binomial Asset Pricing Model.*
- Shreve, S. E. *Stochastic Calculus for Finance II: Continuous-Time Models.*
- Björk, T. *Arbitrage Theory in Continuous Time.*
- Delbaen, F. and Schachermayer, W. *The Mathematics of Arbitrage.*

### Discrete-time and robust extensions

- Cox, J. C., Ross, S. A., and Rubinstein, M. "Option Pricing: A Simplified Approach."
- Harrison, J. M. and Pliska, S. R. "Martingales and Stochastic Integrals in the Theory of Continuous Trading."
- Dolinsky, Y. and Soner, H. M. "Martingale Optimal Transport and Robust Hedging in Continuous Time."
- Beiglböck, M., Henry-Labordère, P., and Penkner, F. "Model-independent bounds for option prices: a mass transport approach."
- Buehler, H., Gonon, L., Teichmann, J., and Wood, B. "Deep Hedging."